# 長江杯 2026 — 80GB Colab 自動跑批與團隊隊列 Worker

**架構說明**：
- **程式碼與公開靶點**：由 GitHub (`David-Liv/changjiang-protein-2026`) 自動同步，隊友可隨時提 PR 更新腳本與配置。
- **任務隊列與產出數據（保密）**：存放在 Google Drive `/content/drive/MyDrive/Changjiang_Cup_2026`。
- **隊友如何提交任務**：隊友無需登入你的 Colab 帳號，只需將 JSON 任務單放入網盤 `Changjiang_Cup_2026/jobs/pending/`，本 Notebook 會自動調用 80GB GPU 執行並將結果寫回 `jobs/completed/` 與 `registry/design_registry.csv`。

In [ ]:
#@title 1. 檢查 80GB GPU、掛載 Google Drive 並拉取最新 GitHub 倉庫
import os, subprocess, pathlib
from google.colab import drive

DRIVE_WORKSPACE = "/content/drive/MyDrive/Changjiang_Cup_2026" #@param {type:"string"}
GITHUB_REPO_URL = "https://github.com/David-Liv/changjiang-protein-2026.git" #@param {type:"string"}
REPO_DIR = "/content/changjiang-protein-2026"

print("=== [1/3] GPU 狀態檢查 ===")
!nvidia-smi

print("\n=== [2/3] 掛載 Google Drive ===")
drive.mount('/content/drive')
pathlib.Path(DRIVE_WORKSPACE).mkdir(parents=True, exist_ok=True)

print("\n=== [3/3] 同步 GitHub 最新 Pipeline 代碼 ===")
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}
!python3 pipeline/worker.py --drive-root "{DRIVE_WORKSPACE}" --once --dry-run

In [ ]:
#@title 2. 安裝 BindCraft + PyRosetta + SolubleMPNN 環境（每台新 VM 執行一次）
import os
if not os.path.exists("/content/BindCraft"):
    !git clone https://github.com/martinpacesa/BindCraft /content/BindCraft
    !bash /content/BindCraft/install_bindcraft.sh --cuda '12.4' --pkg_manager 'conda'
else:
    print("BindCraft 已安裝 ✓")

In [ ]:
#@title 3. 將預設任務模板複製到網盤 `jobs/pending/`（可選：快速開始靶點 5 或靶點 1）
import shutil, pathlib

TEMPLATE_CHOICE = "target5_TL1A_minibinder" #@param ["target5_TL1A_minibinder", "target1_EGFR_vhh", "none"]
pending_dir = pathlib.Path(DRIVE_WORKSPACE) / "jobs" / "pending"
pending_dir.mkdir(parents=True, exist_ok=True)

mapping = {
    "target5_TL1A_minibinder": "/content/changjiang-protein-2026/jobs/templates/job_target5_TL1A_minibinder.json",
    "target1_EGFR_vhh": "/content/changjiang-protein-2026/jobs/templates/job_target1_EGFR_vhh.json",
}
if TEMPLATE_CHOICE in mapping:
    src = pathlib.Path(mapping[TEMPLATE_CHOICE])
    dst = pending_dir / src.name
    shutil.copy2(src, dst)
    print(f"已投遞任務單至隊列：{dst}")

In [ ]:
#@title 4. 啟動 80GB 隊列監聽器（自動拉取 GitHub 更新 + 驗證輸入 + 執行任務 + 寫回總表）
!git -C /content/changjiang-protein-2026 pull --ff-only
!python3 /content/changjiang-protein-2026/pipeline/worker.py --drive-root "{DRIVE_WORKSPACE}" --once